In [ ]:
import heapq

# Added corridor: C -> G. All costs are nonnegative.
weighted_graph = {
    "A": [("B", 1), ("C", 5)],
    "B": [("D", 4), ("E", 1)],
    "C": [("F", 2), ("G", 7)],
    "D": [], "E": [("G", 1)], "F": [], "G": []
}

def ucs(start, goal):
    frontier = [(0, start, [start])]
    best = {start: 0}
    order = []
    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if cost != best.get(node):
            continue
        order.append(node)
        if node == goal:
            return path, cost, order
        for nxt, edge_cost in weighted_graph[node]:
            new_cost = cost + edge_cost
            if new_cost < best.get(nxt, float("inf")):
                best[nxt] = new_cost
                heapq.heappush(frontier, (new_cost, nxt, path + [nxt]))
    return None, None, order

def path_cost(path):
    total = 0
    for a, b in zip(path, path[1:]):
        total += next(cost for nxt, cost in weighted_graph[a] if nxt == b)
    return total

path, cost, order = ucs("A", "G")
bfs_route = ["A", "C", "G"]
print("UCS path:", path)
print("UCS processing order:", order)
print("UCS returned cost:", cost)
print("UCS independently checked cost:", path_cost(path))
print("BFS route:", bfs_route)
print("BFS hop count:", len(bfs_route)-1)
print("BFS total cost:", path_cost(bfs_route))


In [ ]:
graph = {
    "A": ["B", "C"], "B": ["D", "E"], "C": ["F"],
    "D": [], "E": ["G"], "F": [], "G": []
}

def dls(node, goal, limit, path):
    if node == goal:
        return "FOUND", path
    depth = len(path) - 1
    if depth == limit:
        return ("CUTOFF", None) if graph[node] else ("FAILURE", None)

    cutoff = False
    for nxt in graph[node]:
        if nxt not in path:  # path-local cycle check
            status, result = dls(nxt, goal, limit, path + [nxt])
            if status == "FOUND":
                return status, result
            if status == "CUTOFF":
                cutoff = True
    return ("CUTOFF", None) if cutoff else ("FAILURE", None)

def iddfs(start, goal, max_depth):
    tried = []
    for limit in range(max_depth + 1):
        tried.append(limit)
        status, path = dls(start, goal, limit, [start])
        if status == "FOUND":
            return status, path, tried
    return "CUTOFF", None, tried

print("DLS")
for limit in [1, 2, 3, 4]:
    print("Limit", limit, ":", dls("A", "G", limit, ["A"]))
print("\\nIDDFS")
for maximum in [1, 2, 3, 4]:
    print("Maximum", maximum, ":", iddfs("A", "G", maximum))


In [ ]:
from collections import deque

graph = {
    "A": ["B", "C"], "B": ["D", "E"], "C": ["F"],
    "D": [], "E": ["G"], "F": [], "G": []
}

def bfs(start, goal):
    queue = deque([(start, [start])])
    seen = {start}
    order = []
    while queue:
        node, path = queue.popleft()
        order.append(node)
        if node == goal:
            return path, order
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, path + [nxt]))
    return None, order

def dfs(start, goal):
    stack = [(start, [start])]
    seen = set()
    order = []
    while stack:
        node, path = stack.pop()
        if node in seen:
            continue
        seen.add(node)
        order.append(node)
        if node == goal:
            return path, order
        for nxt in reversed(graph[node]):
            if nxt not in seen:
                stack.append((nxt, path + [nxt]))
    return None, order

print("BFS path:", bfs("A", "G")[0])
print("BFS visit order:", bfs("A", "G")[1])
print("DFS path:", dfs("A", "G")[0])
print("DFS visit order:", dfs("A", "G")[1])
